# R Aplicado à Vigilância em Saúde: Dengue e Ilhas de Calor em BH



## Notebook para Google Colab utilizando ambiente de processamento R.

####**Objetivo:** Análise integrada de dengue e clima em Belo Horizonte utilizando IA generativa

####**Formato:** Material auto-instrucional que deve ser preenchido com apoio dos conteúdos, exemplos e snippets disponíveis no reposítório <https://github.com/prisnormando/rvigilanciasaude>.




---

## Instruções Gerais

O notebook está parcialmente preenchido. Para cada etapa a tarefas que necessitam ser cumpridas para que o boletim seja exportado com sucesso ao final.

Siga as instruções propostas uma a uma na sequência em que estão postas (passo-a-passo), realizando todos os ajustes solicitados para cada uma delas. Em caso de dúvidas, a qualquer momento da atividade, você pode usar o [NotebookLM](https://notebooklm.google.com/notebook/c51ea0cd-7a64-4c44-8d3e-9f339c5577c0), o [agente do curso](https://gemini.google.com/gem/ebaaa16f310d?usp=sharing) ou consultar o repositório no [Github](https://github.com/prisnormando/rvigilanciasaude).

Lembre-se de não apagar as saídas de código e, ao final, exportar o notebook com elas para comprovação das atividades de vocês.

As etapas de trabalho são as seguintes:

0. Configuração inicial do Google Colab
    
    - configurar o ambiente, instalar e carregar os pacotes necessários

1. Aquisição dos dados

    - Realizar o download dos dados sobre Dengue, Clima e IVS

2. Processamento e limpeza

    - Processar e manipular os dados para que estejam prontos para análise

3. Análise descritiva

    - Descrever os dados adequadamente

4. Análise epidemiológica

    - Cálcular os indicadores, ajustar a semana epidemiológica, territorializar os dados

5. Visualizações

    - Criar as visualizações sobre as descrições e análises realizadas quando necessário

6. Interpretação com IA

    - Conectar o notebook com a api do Gemini e utilizar a ferramenta para colaborar na interpretação e automatização o boletim

7. Exportação do Boletim

    - Gerar e exportar o relatório em formato de boletim

## ETAPA 0: Configuração Inicial do Google Colab

#### Vamos instalar os pacotes R necessários e configurar o ambiente.

### Instruções:

1 - Ajuste o Ambiente de execução para operar na linguagem R.

2 - Leia a documentação dos pacotes e do curso, inicie uma celula de texto e faça a descrição do que cada um deles faz com suas funções principais. Se necessário você pode usar o [NotebookLM](https://notebooklm.google.com/notebook/c51ea0cd-7a64-4c44-8d3e-9f339c5577c0) e o [agente do curso](https://gemini.google.com/gem/ebaaa16f310d?usp=sharing).

3 - O código na célula abaixo instala o primeiro conjunto de pacotes para realizarmos as tarefas propostas. Inicialmente ele deve funcionar, porém, não está escrito da melhor forma possível. Reescreva de forma a que ele possa conferir se o pacote já existe no ambiente, se há pacotes que estão sendo instalados em duplicidade e se há pacotes faltando para que tudo seja instalado e "rode" adequadamente.


In [ ]:
# Instalar pacotes R necessários (executar uma única vez)
install.packages(c(
  "tidyverse", "lubridate", "janitor", "gtsummary", "corrr", "broom",
  "sf", "tmap", "geobr", "knitr", "kableExtra", "httr", "jsonlite",
  "microdatasus"
), repos = "http://cran.r-project.org", quiet = TRUE)

cat("✓ Pacotes instalados com sucesso!\n")

4 - Carregue os pacotes

In [ ]:
# Carregar bibliotecas
library(tidyverse)
library(lubridate)
library(janitor)
library(gtsummary)
library(corrr)
library(broom)
library(sf)
library(geobr)
library(knitr)
library(kableExtra)
library(httr)
library(jsonlite)

cat("✓ Bibliotecas carregadas!\n")

5 - O código abaixo carrega a pasta de trabalho dentro do ambiente efêmero do Google Colab, reescreva o código de forma que o script:

    5.1 - inicie o Google Drive no ambiente
    5.2 - as pastas sigam as zonas de dados raw, transientes e confiáveis; docs; boletim
    5.3 - ajuste o ambiente para sempre apontar para as pastas no Google Drive

In [ ]:
# Criar diretórios de trabalho
dir.create("/content/dados_processados", showWarnings = FALSE)
dir.create("/content/resultados_analise", showWarnings = FALSE)
dir.create("/content/boletim_final", showWarnings = FALSE)

cat("✓ Diretórios criados!\n")

## ETAPA 1: Download e Processamento de Dados SINAN (Dengue)

#### Vamos baixar os dados de dengue do SINAN via OpenDataSUS para Belo Horizonte.

### Instruções:

6 - O código abaixo cria uma base sintética com 5000 casos de dengue para a cidade de Belo Horizonte - MG. Esse método é muito bom para testarmos a conceito do script, mas não funciona em produção. Utilize o pacote microdatasus e baixe os dados para 2024. Lembre-se:

    - Estamos trabalhando apenas com Belo Horizonte;
    - As bases do SINAN e os dados de clima serão pareados, então é preciso fazer a seleção de variáveis compatível com os dados climáticos e a temática do boletim;
    - Normalmente é necessário realizar a limpeza e outras atividades de pré-processamento para que tudo corra bem nas próximas etapas;
    - Sugiro que você leia e entenda o código das Etapas 2 e 3 para garantir que está baixando os dados adequados.

7 - Adapte o código para que as células estejam dedicadas apenas a uma função. No caso da célula abaixo, ela está criando a escrevendo o dataset ao mesmo tempo. Separe as células quando for baixar os dados, carregar o dataframe, escrever o dataset de trabalho e conferir o dataset. Lembre-se que o dataset deve ser escrito dentro do Google Drive.


In [ ]:
cat("\n=== ETAPA 1: Download SINAN Dengue ===")
cat("\nBaixando dados de dengue do SINAN (OpenDataSUS)...\n")

# Usar a API do OpenDataSUS para baixar dados de dengue
# Nota: Este é um exemplo simplificado. Em produção, usar microdatasus::fetch_datasus()

# Para este notebook, vamos criar dados simulados realistas
set.seed(42)

# Criar dataset simulado de dengue em BH
n_casos <- 5000

sinan_dengue <- tibble(
  data_notificacao = seq(as.Date("2023-01-01"), as.Date("2024-12-31"), length.out = n_casos),
  sexo = sample(c("M", "F"), n_casos, replace = TRUE, prob = c(0.45, 0.55)),
  idade = sample(0:90, n_casos, replace = TRUE, prob = dnorm(0:90, mean = 35, sd = 20)),
  classificacao = sample(
    c("Dengue Clássica", "Dengue Grave", "Febre Hemorrágica"),
    n_casos, replace = TRUE, prob = c(0.85, 0.12, 0.03)
  ),
  evolucao = sample(
    c("Cura", "Óbito"),
    n_casos, replace = TRUE, prob = c(0.995, 0.005)
  ),
  bairro = sample(
    c("Centro", "Pampulha", "Barreiro", "Venda Nova", "Nordeste", "Noroeste", "Oeste", "Sul"),
    n_casos, replace = TRUE
  )
)

# Salvar dados
write_csv(sinan_dengue, "/content/dados_processados/sinan_dengue_bh_microdados.csv")

cat("✓ Dados SINAN baixados e salvos!\n")
cat("  Total de casos:", nrow(sinan_dengue), "\n")
cat("  Período:", min(sinan_dengue$data_notificacao), "a", max(sinan_dengue$data_notificacao), "\n")

# Visualizar primeiras linhas
head(sinan_dengue) %>% kable()

7.1 - Escreva o script que confere o cabeçalho, as primeiras 20 linhas do dataframe, as 20 linhas centrais do dataframe e as últimas 20 linhas do dataframe.

7.2 - Agregue os dados do dataframe por semana epidemiológica e crie um novo dataset, escrevendo-o no Google Drive.

## ETAPA 2: Base de Dados Climáticos (BR-DWGD)

#### Vamos criar dados climáticos simulados realistas para Belo Horizonte.

### Instruções:

8 - O script abaixo gera uma base sintética com dados climáticos. Neste caso, vamos manter o método por conta da capacidade do ambiente de processamento. Separe o código em etapas, abra uma célula de texto acima de cada trecho de código e documente explicando o que o código executa. Lembre-se de conferir se os dados estão sendo carregados para dentro do Google Drive.

In [ ]:
cat("\n=== ETAPA 2: Dados Climáticos ===")
cat("\nCriando dados climáticos para Belo Horizonte...\n")

# Criar dados climáticos diários (simulados realistas)
datas <- seq(as.Date("2023-01-01"), as.Date("2024-12-31"), by = "day")

clima_diario <- tibble(
  data = datas,
  tmax = 25 + 8 * sin(as.numeric(datas) / 365 * 2 * pi) + rnorm(length(datas), 0, 2),
  tmin = 15 + 6 * sin(as.numeric(datas) / 365 * 2 * pi) + rnorm(length(datas), 0, 1.5),
  precipitacao = pmax(0, 5 + 10 * sin(as.numeric(datas) / 365 * 2 * pi) + rnorm(length(datas), 0, 3))
) %>%
  mutate(
    semana_epi = epiweek(data),
    ano = year(data)
  )

# Agregar por semana epidemiológica
clima_semanal <- clima_diario %>%
  group_by(ano, semana_epi) %>%
  summarise(
    data = min(data),
    tmax_media = mean(tmax, na.rm = TRUE),
    tmin_media = mean(tmin, na.rm = TRUE),
    precipitacao_total = sum(precipitacao, na.rm = TRUE),
    .groups = "drop"
  )

write_csv(clima_semanal, "/content/dados_processados/clima_semanal_bh.csv")

cat("✓ Dados climáticos criados e salvos!\n")
cat("  Total de semanas:", nrow(clima_semanal), "\n")

head(clima_semanal) %>% kable()

## ETAPA 3: Integração de Dados (Dengue + Clima)

#### Vamos integrar os dados de dengue e clima, criando defasagens temporais (lags).

### Instruções

9 - Faça a separação das células e documentação do código, conforme já realizado em outras etapas.

10 - Refatore o código para que as variáveis do SINAN possam ser pareadas com a base de clima criada sinteticamente.

In [ ]:
cat("\n=== ETAPA 3: Integração Dengue + Clima ===")
cat("\nIntegrando dados e criando defasagens temporais...\n")

# Agregar casos de dengue por semana epidemiológica
dengue_semanal <- sinan_dengue %>%
  mutate(
    semana_epi = epiweek(data_notificacao),
    ano = year(data_notificacao)
  ) %>%
  group_by(ano, semana_epi) %>%
  summarise(
    total_casos = n(),
    casos_graves = sum(classificacao %in% c("Dengue Grave", "Febre Hemorrágica")),
    obitos = sum(evolucao == "Óbito"),
    .groups = "drop"
  )

# Integrar com dados climáticos
dataset_integrado <- dengue_semanal %>%
  left_join(clima_semanal, by = c("ano", "semana_epi")) %>%
  arrange(ano, semana_epi) %>%
  # Criar defasagens (lags)
  mutate(
    tmax_lag1 = lag(tmax_media, 1),
    tmax_lag2 = lag(tmax_media, 2),
    tmax_lag3 = lag(tmax_media, 3),
    tmax_lag4 = lag(tmax_media, 4),
    precipitacao_lag1 = lag(precipitacao_total, 1),
    precipitacao_lag2 = lag(precipitacao_total, 2)
  ) %>%
  drop_na()

write_csv(dataset_integrado, "/content/dados_processados/dataset_boletim_dengue_clima_bh.csv")

cat("✓ Dados integrados com sucesso!\n")
cat("  Total de semanas:", nrow(dataset_integrado), "\n")

head(dataset_integrado) %>% kable()

10.1 - Escreva o script que confere o cabeçalho, as primeiras 20 linhas do dataframe, as 20 linhas centrais do dataframe e as últimas 20 linhas do dataframe.

## ETAPA 4: Análise Descritiva

#### Vamos gerar tabelas descritivas dos casos de dengue.

### Instruções

11 - Refatore o código separando as tabelas e salvando uma a uma com a devida documentação.

In [ ]:
cat("\n=== ETAPA 4: Análise Descritiva ===")
cat("\nGerando estatísticas descritivas...\n")

# Análise por sexo
cat("\n--- Distribuição por Sexo ---\n")
tabela_sexo <- sinan_dengue %>%
  group_by(sexo) %>%
  summarise(
    casos = n(),
    percentual = round(n() / nrow(sinan_dengue) * 100, 1),
    casos_graves = sum(classificacao %in% c("Dengue Grave", "Febre Hemorrágica")),
    obitos = sum(evolucao == "Óbito")
  )

print(tabela_sexo)

# Análise por faixa etária
cat("\n--- Distribuição por Faixa Etária ---\n")
tabela_idade <- sinan_dengue %>%
  mutate(
    faixa_etaria = cut(idade, breaks = c(0, 10, 20, 40, 60, 100),
                       labels = c("0-10", "11-20", "21-40", "41-60", "60+"))
  ) %>%
  group_by(faixa_etaria) %>%
  summarise(
    casos = n(),
    percentual = round(n() / nrow(sinan_dengue) * 100, 1),
    casos_graves = sum(classificacao %in% c("Dengue Grave", "Febre Hemorrágica")),
    obitos = sum(evolucao == "Óbito")
  )

print(tabela_idade)

# Análise por classificação
cat("\n--- Distribuição por Classificação ---\n")
tabela_classificacao <- sinan_dengue %>%
  group_by(classificacao) %>%
  summarise(
    casos = n(),
    percentual = round(n() / nrow(sinan_dengue) * 100, 1),
    obitos = sum(evolucao == "Óbito")
  )

print(tabela_classificacao)

# Salvar tabelas
write_csv(tabela_sexo, "/content/resultados_analise/tabela_sexo.csv")
write_csv(tabela_idade, "/content/resultados_analise/tabela_idade.csv")
write_csv(tabela_classificacao, "/content/resultados_analise/tabela_classificacao.csv")

cat("\n✓ Análise descritiva concluída!\n")

## ETAPA 5: Análise Epidemiológica

#### Vamos calcular indicadores epidemiológicos e correlações.

### Instruções:

12 - Refatore as células, separando o cálculo dos indicadores dos das correlações.

In [ ]:
cat("\n=== ETAPA 5: Análise Epidemiológica ===")
cat("\nCalculando indicadores epidemiológicos...\n")

# População de BH (IBGE 2022)
populacao_bh <- 2315560

# Indicadores por ano
indicadores <- sinan_dengue %>%
  mutate(ano = year(data_notificacao)) %>%
  group_by(ano) %>%
  summarise(
    casos_totais = n(),
    casos_graves = sum(classificacao %in% c("Dengue Grave", "Febre Hemorrágica")),
    obitos_totais = sum(evolucao == "Óbito"),
    .groups = "drop"
  ) %>%
  mutate(
    taxa_incidencia = round((casos_totais / populacao_bh) * 100000, 2),
    proporcao_graves = round((casos_graves / casos_totais) * 100, 2),
    taxa_letalidade = round((obitos_totais / casos_totais) * 100, 2)
  )

cat("\n--- Indicadores Epidemiológicos Anuais ---\n")
print(indicadores)

write_csv(indicadores, "/content/resultados_analise/indicadores_epidemiologicos_anuais.csv")

# Análise de correlação (Dengue vs Clima)
cat("\n--- Correlação entre Dengue e Temperatura (Lags) ---\n")

correlacoes <- dataset_integrado %>%
  select(total_casos, tmax_media, tmax_lag1, tmax_lag2, tmax_lag3, tmax_lag4) %>%
  correlate(method = "pearson") %>%
  focus(total_casos) %>%
  arrange(desc(abs(total_casos)))

print(correlacoes)

write_csv(correlacoes, "/content/resultados_analise/correlacoes_clima_dengue.csv")

cat("\n✓ Análise epidemiológica concluída!\n")

## ETAPA 6: Visualizações

#### Vamos criar gráficos para visualizar os dados.

### Instruções:

13 - Rode o script e faça as adaptações necessárias para que as imagens estejam legíveis e informativas.

14 - Verifique se elas estão sendo salvas no local correto, dentro do sistema de arquivos.

In [ ]:
cat("\n=== ETAPA 6: Visualizações ===")
cat("\nGerando gráficos...\n")

# Gráfico 1: Série Temporal de Dengue
p1 <- dataset_integrado %>%
  ggplot(aes(x = data, y = total_casos)) +
  geom_line(color = "#c0392b", size = 0.8) +
  geom_point(color = "#c0392b", size = 1) +
  labs(
    title = "Série Temporal: Casos de Dengue em Belo Horizonte",
    x = "Data", y = "Casos por Semana"
  ) +
  theme_minimal() +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))

print(p1)
ggsave("/content/resultados_analise/grafico_serie_casos_dengue.png", p1, width = 10, height = 6)

# Gráfico 2: Distribuição por Faixa Etária
p2 <- sinan_dengue %>%
  mutate(
    faixa_etaria = cut(idade, breaks = c(0, 10, 20, 40, 60, 100),
                       labels = c("0-10", "11-20", "21-40", "41-60", "60+"))
  ) %>%
  group_by(faixa_etaria) %>%
  summarise(casos = n()) %>%
  ggplot(aes(x = faixa_etaria, y = casos, fill = faixa_etaria)) +
  geom_col() +
  labs(
    title = "Distribuição de Casos por Faixa Etária",
    x = "Faixa Etária", y = "Número de Casos"
  ) +
  theme_minimal() +
  theme(legend.position = "none", plot.title = element_text(hjust = 0.5, face = "bold"))

print(p2)
ggsave("/content/resultados_analise/grafico_faixa_etaria.png", p2, width = 8, height = 6)

# Gráfico 3: Correlação Dengue vs Temperatura (Lag 2)
p3 <- dataset_integrado %>%
  ggplot(aes(x = tmax_lag2, y = total_casos)) +
  geom_point(color = "#e74c3c", size = 2, alpha = 0.6) +
  geom_smooth(method = "lm", se = TRUE, color = "#2c3e50") +
  labs(
    title = "Correlação: Dengue vs Temperatura (Lag 2 semanas)",
    x = "Temperatura Máxima (°C) - 2 semanas antes",
    y = "Casos de Dengue"
  ) +
  theme_minimal() +
  theme(plot.title = element_text(hjust = 0.5, face = "bold"))

print(p3)
ggsave("/content/resultados_analise/grafico_correlacao_temp_lag2.png", p3, width = 10, height = 6)

cat("\n✓ Gráficos gerados e salvos!\n")

## ETAPA 7: Integração com Google Gemini (IA Generativa)

#### Vamos usar a API do Google Gemini para gerar interpretações automáticas dos dados.

### Instruções:

15 - Vá até o repositório do curso em <https://github.com/prisnormando/rvigilanciasaude/tree/main/scripts/8.%20llms> e em <https://github.com/prisnormando/rvigilanciasaude/blob/main/ebooks/comoUsarLlmsNoR.md>. A partir desse material de consulta, adapte o código abaixo com todas as suas funcionalidades para funcionar com o pacote "gemini.R". Lembre-se que será necessário instalar e carregar o pacote.

16 - Caso os resultados não estejam sendo satisfatórios, adapte os prompts disponíveis no script original.

**Obs:** Mantenha a consistência quanto a separação de células e a documentação do código.

In [ ]:
cat("\n=== ETAPA 7: Integração com Google Gemini ===")
cat("\nConfigurando acesso à API do Gemini...\n")

# Função para chamar a API do Gemini
chamar_gemini <- function(prompt_text, api_key = Sys.getenv("GEMINI_API_KEY")) {

  if (api_key == "") {
    cat("\n⚠️  AVISO: Chave da API do Gemini não configurada.\n")
    cat("Para usar IA generativa, configure a variável de ambiente GEMINI_API_KEY.\n")
    cat("\nUsando texto de fallback...\n")
    return("[Texto gerado por IA - Configure GEMINI_API_KEY para ativar]")
  }

  url <- "https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-pro:generateContent"

  body <- list(
    contents = list(list(parts = list(list(text = prompt_text)))),
    generationConfig = list(
      temperature = 0.2,
      maxOutputTokens = 1024
    )
  )

  tryCatch({
    response <- POST(
      paste0(url, "?key=", api_key),
      body = toJSON(body, auto_unbox = TRUE),
      add_headers("Content-Type" = "application/json")
    )

    if (status_code(response) == 200) {
      return(content(response, "parsed")$candidates[[1]]$content$parts[[1]]$text)
    } else {
      return(paste("Erro na API:", status_code(response)))
    }
  }, error = function(e) {
    return(paste("Erro ao chamar API:", e$message))
  })
}

cat("\n✓ Função de integração com Gemini configurada!\n")
cat("\nNota: Para usar a IA generativa, configure GEMINI_API_KEY antes de executar.\n")

In [ ]:
# Gerar interpretação descritiva com IA
cat("\nGerando interpretação descritiva com IA...\n")

prompt_descritivo <- paste0(
  "Você é um epidemiologista experiente analisando dados de dengue em Belo Horizonte.\n",
  "\nDados:\n",
  "- Total de casos: ", nrow(sinan_dengue), "\n",
  "- Casos graves: ", sum(sinan_dengue$classificacao %in% c("Dengue Grave", "Febre Hemorrágica")), "\n",
  "- Óbitos: ", sum(sinan_dengue$evolucao == "Óbito"), "\n",
  "- Taxa de incidência: ", round((nrow(sinan_dengue) / 2315560) * 100000, 1), " por 100k\n",
  "- Proporção de mulheres: ", round(sum(sinan_dengue$sexo == "F") / nrow(sinan_dengue) * 100, 1), "%\n",
  "\nEscreva um parágrafo curto (3-4 frases) interpretando estes dados.\n",
  "Destaque os grupos mais afetados e a gravidade da epidemia."
)

texto_descritivo <- chamar_gemini(prompt_descritivo)

cat("\n--- Interpretação Descritiva (IA) ---\n")
cat(texto_descritivo, "\n")

writeLines(texto_descritivo, "/content/resultados_analise/texto_gemini_descritiva.txt")

In [ ]:
# Gerar interpretação epidemiológica com IA
cat("\nGerando interpretação epidemiológica com IA...\n")

# Encontrar o melhor lag
melhor_lag <- correlacoes %>%
  filter(term != "total_casos") %>%
  slice(1) %>%
  pull(term)

melhor_correlacao <- correlacoes %>%
  filter(term != "total_casos") %>%
  slice(1) %>%
  pull(total_casos)

prompt_epidemiologico <- paste0(
  "Você é um Analista Epidemiológico Especialista em Mudanças Climáticas.\n",
  "\nAnalisamos os dados de dengue e clima em Belo Horizonte.\n",
  "Descobrimos que a correlação mais forte (r=", round(melhor_correlacao, 2), ") ocorre com: ", melhor_lag, "\n",
  "\nEscreva um alerta epidemiológico curto (1 parágrafo) explicando:\n",
  "1. O que essa defasagem (lag) significa no ciclo do mosquito\n",
  "2. Implicações práticas para controle vetorial\n",
  "3. Uma recomendação de ação baseada neste achado"
)

texto_epidemiologico <- chamar_gemini(prompt_epidemiologico)

cat("\n--- Interpretação Epidemiológica (IA) ---\n")
cat(texto_epidemiologico, "\n")

writeLines(texto_epidemiologico, "/content/resultados_analise/texto_gemini_epi.txt")

## ETAPA 8: Geração do Relatório Final

#### Vamos compilar um relatório resumido com todos os achados.

17 - Refatore o código abaixo para que ele possa ser exportado para HTML e PDF além do Markdown.

    Procure usar elementos como ícones, logos e cores para deixar seu relatório esteticamente agradável.

In [ ]:
cat("\n=== ETAPA 8: Relatório Final ===")
cat("\nCompilando relatório resumido...\n")

relatorio <- paste0(
  "# BOLETIM EPIDEMIOLÓGICO\n",
  "## Dengue e Ilhas de Calor em Belo Horizonte\n",
  "**Data:** ", Sys.Date(), "\n",
  "**Período de Análise:** 2023-2024\n\n",

  "### 1. INDICADORES GERAIS\n",
  "- **Total de Casos:** ", nrow(sinan_dengue), "\n",
  "- **Casos Graves:** ", sum(sinan_dengue$classificacao %in% c("Dengue Grave", "Febre Hemorrágica")), "\n",
  "- **Óbitos:** ", sum(sinan_dengue$evolucao == "Óbito"), "\n",
  "- **Taxa de Incidência:** ", round((nrow(sinan_dengue) / 2315560) * 100000, 1), " por 100.000 hab.\n\n",

  "### 2. DISTRIBUIÇÃO DEMOGRÁFICA\n",
  "- **Mulheres:** ", round(sum(sinan_dengue$sexo == "F") / nrow(sinan_dengue) * 100, 1), "%\n",
  "- **Homens:** ", round(sum(sinan_dengue$sexo == "M") / nrow(sinan_dengue) * 100, 1), "%\n",
  "- **Idade Média:** ", round(mean(sinan_dengue$idade), 1), " anos\n\n",

  "### 3. CORRELAÇÃO COM CLIMA\n",
  "- **Melhor Preditor:** ", melhor_lag, "\n",
  "- **Correlação (r):** ", round(melhor_correlacao, 3), "\n",
  "- **Interpretação:** Forte correlação entre temperatura e casos de dengue com defasagem de 2 semanas.\n\n",

  "### 4. INTERPRETAÇÃO DA IA\n",
  texto_descritivo, "\n\n",
  texto_epidemiologico, "\n\n",

  "### 5. RECOMENDAÇÕES\n",
  "1. Intensificar ações de controle vetorial após períodos de calor intenso.\n",
  "2. Preparar rede assistencial para aumento de demanda 2-3 semanas após ondas de calor.\n",
  "3. Direcionar campanhas de conscientização para grupos demográficos mais afetados.\n\n",

  "---\n",
  "**Relatório Gerado Automaticamente com R + IA Generativa (Google Gemini)**\n"
)

writeLines(relatorio, "/content/boletim_final/RELATORIO_RESUMIDO.md")

cat(relatorio)

**Desafio**

18 - Escreva um script que exporta o relatório para um HTML autocontido que possa ser exibido em qualquer servidor web.

## ETAPA 9: Resumo e Próximos Passos

Parabéns! Você completou o pipeline completo de análise.

In [ ]:
cat("\n╔════════════════════════════════════════════════════════════════════════╗\n")
cat("║                    PIPELINE CONCLUÍDO COM SUCESSO!                      ║\n")
cat("╚════════════════════════════════════════════════════════════════════════╝\n\n")

cat("✓ ETAPA 1: Download de dados SINAN\n")
cat("✓ ETAPA 2: Processamento de dados climáticos\n")
cat("✓ ETAPA 3: Integração de dados (Dengue + Clima)\n")
cat("✓ ETAPA 4: Análise descritiva\n")
cat("✓ ETAPA 5: Análise epidemiológica\n")
cat("✓ ETAPA 6: Visualizações (gráficos)\n")
cat("✓ ETAPA 7: Integração com Google Gemini (IA)\n")
cat("✓ ETAPA 8: Geração do relatório\n\n")

cat("Arquivos Gerados:\n")
cat("  📁 /content/dados_processados/\n")
cat("     - sinan_dengue_bh_microdados.csv\n")
cat("     - clima_semanal_bh.csv\n")
cat("     - dataset_boletim_dengue_clima_bh.csv\n\n")

cat("  📁 /content/resultados_analise/\n")
cat("     - indicadores_epidemiologicos_anuais.csv\n")
cat("     - correlacoes_clima_dengue.csv\n")
cat("     - grafico_serie_casos_dengue.png\n")
cat("     - grafico_faixa_etaria.png\n")
cat("     - grafico_correlacao_temp_lag2.png\n")
cat("     - texto_gemini_descritiva.txt\n")
cat("     - texto_gemini_epi.txt\n\n")

cat("  📁 /content/boletim_final/\n")
cat("     - RELATORIO_RESUMIDO.md\n\n")

cat("Próximos Passos:\n")
cat("1. Baixe os arquivos gerados\n")
cat("2. Customize o relatório com seu logo e dados institucionais\n")
cat("3. Publique o boletim em seu site ou plataforma de vigilância\n")
cat("4. Agende a execução automática para gerar boletins semanais\n\n")

cat("Tempo Total de Execução: ~5-10 minutos\n")
cat("\n✨ Análise Completa com Sucesso! ✨\n")

## Apêndice: Dicas e Troubleshooting

### Como Usar a API do Gemini (Opcional)

Se quiser ativar as interpretações automáticas com IA:

1. Acesse https://aistudio.google.com/
2. Clique em "Get API key"
3. Copie a chave gerada
4. No Colab, vá em "Secrets" (ícone de chave no menu lateral)
5. Crie um novo secret chamado `GEMINI_API_KEY`
6. Cole a chave
7. Execute novamente as células de IA

### Adaptando para Seus Dados

Para usar dados reais do SINAN:

```r
# Instalar e usar o pacote microdatasus
install.packages("microdatasus")
library(microdatasus)

# Baixar dados reais
sinan_dengue <- fetch_datasus(
  year_start = 2023,
  year_end = 2024,
  uf = "MG",
  disease = "dengue"
)
```
**Fim do Notebook**